# Notebook 02 — Binder Generation & In Silico Evaluation

**Phases 2 & 3** of the Protein Binder Evaluation & RL-Guided Design Pipeline.

This notebook:
1. Generates binder candidates via RFdiffusion + ProteinMPNN (or synthetic fallback)
2. Evaluates candidates with Chai-1/ESMFold (or mock backend)
3. Applies Latent-X-style iPTM/pLDDT/RMSD/PAE filters
4. Produces the full analysis suite from Phase 4

---

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
from pathlib import Path

from src.target_prep import TargetProtein, BENCHMARK_TARGETS
from src.generate import BinderGenerator
from src.evaluate import BinderEvaluator
from src.analyse import BinderAnalyser

import yaml
with open('../configs/eval_thresholds.yaml') as f:
    config = yaml.safe_load(f)

print('Config loaded. Targets:', list(config['targets'].keys()))

## 2.1 Load Target

In [ ]:
TARGET_NAME = 'EGFR'  # Change to 'IL7RA', 'TrkA', etc.

meta = BENCHMARK_TARGETS[TARGET_NAME]
target = TargetProtein(
    pdb_id=meta['pdb_id'],
    chain_id=meta['chain_id'],
)
target.download_pdb()
target.parse_structure()
target.identify_hotspots()

print(target)
print(f'Hotspot residues: {target.hotspot_residue_numbers()[:10]}...')

## 2.2 Generate Binder Candidates

If RFdiffusion/ProteinMPNN are installed, real backbones/sequences will be generated.
Otherwise, synthetic candidates are used for full pipeline testing.

In [ ]:
RFDIFFUSION_DIR = None  # Set to Path('/path/to/RFdiffusion') if installed
PROTEINMPNN_DIR = None  # Set to Path('/path/to/ProteinMPNN') if installed

generator = BinderGenerator(
    target=target,
    output_dir=f'../data/generated/{TARGET_NAME}',
    rfdiffusion_dir=RFDIFFUSION_DIR,
    proteinmpnn_dir=PROTEINMPNN_DIR,
)

print(f'RFdiffusion available: {generator._rfdiffusion_available}')
print(f'ProteinMPNN available: {generator._proteinmpnn_available}')

In [ ]:
# Generate backbones (50 designs = fast demo; scale to 100+ for real runs)
N_DESIGNS = 50
SEQS_PER_BACKBONE = 8

backbones = generator.run_rfdiffusion(
    num_designs=N_DESIGNS,
    diffusion_steps=50,
    binder_length=70,
)
print(f'Generated {len(backbones)} backbones.')

In [ ]:
candidates = generator.run_proteinmpnn(
    backbone_pdbs=backbones,
    seqs_per_backbone=SEQS_PER_BACKBONE,
    temperature=0.1,
)

cand_df = generator.to_dataframe(candidates)
generator.save_candidates(candidates)

print(f'\nTotal candidates: {len(cand_df)}')
print(f'Sequence length stats:')
print(cand_df['binder_length'].describe())
cand_df.head()

## 3.1 In Silico Evaluation (Latent-X Protocol)

In [ ]:
# Load target-specific thresholds
target_thresholds = config['targets'].get(TARGET_NAME, config['defaults'])
thresholds = {
    'iptm_min': target_thresholds.get('iptm_min', 0.55),
    'plddt_min': target_thresholds.get('plddt_min', 70.0),
    'interface_rmsd_max': target_thresholds.get('interface_rmsd_max', 2.0),
    'pae_max': config['defaults'].get('pae_max', 10.0),
}
print('Using thresholds:', thresholds)

In [ ]:
# Set backend='chai1' if chai_lab is installed, 'esmfold' if transformers+GPU,
# 'mock' for fast testing without GPU
BACKEND = 'mock'
N_SEEDS = 5  # Matching Latent-X paper protocol

evaluator = BinderEvaluator(
    target=target,
    backend=BACKEND,
    thresholds=thresholds,
)

print(f'Evaluating {len(candidates)} candidates with backend={BACKEND}, n_seeds={N_SEEDS}...')
results_df = evaluator.evaluate(candidates, n_seeds=N_SEEDS)

# Save results
results_path = f'../data/generated/{TARGET_NAME}/evaluation_results.csv'
results_df.to_csv(results_path, index=False)
print(f'Results saved to {results_path}')

## 3.2 Filter Summary

In [ ]:
passing = results_df[results_df['passes_all']]
print(f'Filter summary for {TARGET_NAME}:')
print(f'  Total candidates: {len(results_df)}')
print(f'  Passing iPTM >= {thresholds["iptm_min"]}: {results_df["passes_iptm"].sum()} ({results_df["passes_iptm"].mean():.1%})')
print(f'  Passing pLDDT >= {thresholds["plddt_min"]}: {results_df["passes_plddt"].sum()} ({results_df["passes_plddt"].mean():.1%})')
print(f'  Passing RMSD <= {thresholds["interface_rmsd_max"]}: {results_df["passes_rmsd"].sum()} ({results_df["passes_rmsd"].mean():.1%})')
print(f'  Passing PAE <= {thresholds["pae_max"]}: {results_df["passes_pae"].sum()} ({results_df["passes_pae"].mean():.1%})')
print(f'  PASSING ALL: {len(passing)} ({evaluator.hit_rate(results_df):.1%})')
print(f'\nTop 5 candidates by iPTM:')
print(results_df.nsmallest(5, 'rank')[['rank', 'iptm', 'plddt_binder', 'interface_rmsd', 'pae_inter', 'sequence']].to_string())

## 4.1 Phase 4 Analysis — Full Suite

In [ ]:
from pathlib import Path
figures_dir = Path(f'../figures/{TARGET_NAME}')
figures_dir.mkdir(parents=True, exist_ok=True)

analyser = BinderAnalyser(
    results_df=results_df,
    output_dir=figures_dir,
    target_name=TARGET_NAME,
)

In [ ]:
fig = analyser.plot_hit_rate_summary(save=True)
plt.show()

In [ ]:
fig = analyser.plot_metric_distributions(save=True)
plt.show()

In [ ]:
fig = analyser.plot_secondary_structure_composition(save=True)
plt.show()

In [ ]:
# Sensitivity analysis: how does hit rate vary with iPTM threshold?
fig = analyser.plot_threshold_sweep('iptm', save=True)
plt.show()

In [ ]:
fig = analyser.plot_threshold_sweep('plddt_binder', save=True)
plt.show()

In [ ]:
try:
    fig = analyser.plot_metric_correlations(save=True)
    plt.show()
except Exception as e:
    print(f'Correlation plot skipped: {e}')

In [ ]:
report = analyser.generate_report()

## 4.2 Top Binders Visualisation

In [ ]:
top5 = results_df.nsmallest(5, 'rank')

try:
    import py3Dmol
    import pathlib
    
    for _, row in top5.iterrows():
        backbone_path = pathlib.Path(row['backbone_pdb'])
        if backbone_path.exists():
            view = py3Dmol.view(width=600, height=400)
            view.addModel(backbone_path.read_text(), 'pdb')
            view.setStyle({'cartoon': {'color': 'spectrum'}})
            view.zoomTo()
            print(f"Rank {row['rank']} | iPTM={row['iptm']:.3f} | {row['sequence'][:20]}...")
            view.show()
            break  # Show first one for demo
except ImportError:
    print('py3Dmol not installed.')
    print('Top 5 binder sequences:')
    for _, row in top5.iterrows():
        print(f"  Rank {row['rank']:2d} | iPTM={row['iptm']:.3f} | pLDDT={row['plddt_binder']:.1f} | {row['sequence'][:30]}...")

## Interview Talking Points

**Q: Why does joint sequence-structure generation matter?**

The two-step RFdiffusion → ProteinMPNN pipeline separates backbone design from sequence design. This means the sequence is optimised for a given backbone, not jointly. Latent-X's joint approach should produce better energy landscapes because the sequence and structure co-evolve. We see this in the metric distributions above — the RMSD between generated and re-predicted poses is a direct measure of this consistency.

**Q: How do you evaluate a new model version?**

Run it through this exact pipeline: same targets, same filter thresholds, same n_seeds=5, compare hit rates and iPTM distributions. The threshold sweep curves let you characterise the precision-recall tradeoff.

**Q: What are the failure modes of iPTM-based filtering?**

The sensitivity analysis (threshold sweep) reveals this directly. At iPTM > 0.7, very few designs pass but they're highly confident. At iPTM > 0.4, we get many false positives. The optimal threshold is target-dependent.